In [1]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive to Server...' from Ctrl+Shift+P, then re-run this cell)
Mounted at /content/drive
artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
Colab bootstrap OK. Working directory: /content/projecto_rag


In [33]:
import re
import unicodedata
import yaml
import environment
import store
from console import StageRun
from schema import Document, DocMetadata, Section, documents_hash, validate_documents

# 1. Inicializamos entorno
ART = environment.artifacts_dir(False)
run = StageRun.start("01_documents", ART, profile="full")

legal_root = environment.corpora_dir() / "legalize-ar"
store.fetch_git("https://github.com/legalize-dev/legalize-ar", legal_root, branch="main", commit="3d2450b")

palabras_clave = ["defensa del consumidor", "24.240", "ley 24240"]
subconjunto = sorted(
    [
        p for p in legal_root.rglob("*.md")
        if any(
            kw in p.read_text(encoding="utf-8").lower()
            for kw in palabras_clave
        )
    ],
    key=lambda p: str(p.relative_to(legal_root))
)

# 2. Contadores obligatorios para el informe
CLEANING_HITS: dict[str, int] = {}
CLEANING_CHARS: dict[str, int] = {}

def apply(rule: str, fn, text: str) -> str:
    out = fn(text)
    if out != text:
        CLEANING_HITS[rule] = CLEANING_HITS.get(rule, 0) + 1
        CLEANING_CHARS[rule] = CLEANING_CHARS.get(rule, 0) + (len(text) - len(out))
    return out

def norm_nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def norm_spaces(text: str) -> str:
    lines = [" ".join(line.split()) for line in text.splitlines()]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()

regex_articulo = re.compile(
    r"^[ \t]*(?:#{1,6}[ \t]*|\*{2})"
    r"(?:ART[ÍI]CULO|ART\.)[ \t]+"
    r"(\d+)(?:[º°])?"
    r"[ \t]*"
    r"(?:(?:\*{2})?[ \t]*(?:\.[ \t]*-|\.|—|–|-)"
    r"|(?:\.[ \t]*-|\.|—|–|-)[ \t]*\*{2})"
    r"(?![0-9)])",
    re.MULTILINE | re.IGNORECASE
)

regex_anexo = re.compile(
    r"^[ \t]*(?:"
    r"#{1,6}[ \t]+ANEXO[ \t]+"
    r"([IVXLCDM]+[a-z]?|\d+[a-z]?)"
    r"[ \t]*$"
    r"|"
    r"\*\*ANEXO[ \t]*(?:\n[ \t]*){0,2}"
    r"([IVXLCDM]+[a-z]?|\d+[a-z]?)"
    r"[ \t]*\*\*"
    r")",
    re.MULTILINE | re.IGNORECASE
)

docs_finales = []



def reparar_caracteres_control(texto: str) -> str:
    return texto.translate(str.maketrans({
        "\x95": "•",
        "\x96": "–",
        "\x97": "—"
    }))


# 4. Procesamiento
for path in subconjunto:
    texto_crudo = path.read_text(encoding="utf-8")

    texto_nfc = apply(
        "NFC_Acentos",
        norm_nfc,
        texto_crudo
    )

    texto_nfc = apply(
        "Caracteres_Control",
        reparar_caracteres_control,
        texto_nfc
    )

    if texto_nfc.startswith("---"):
        partes = texto_nfc.split("---", 2)
        if len(partes) >= 3:
            try:
              metadatos = yaml.safe_load(partes[1]) or {}
            except yaml.YAMLError as e:
              raise ValueError(
                  f"Error al leer los metadatos YAML de {path.name}: {e}"
              ) from e
            contenido = partes[2]
        else:
            metadatos = {}
            contenido = texto_nfc
    else:
        metadatos = {}
        contenido = texto_nfc

    ley_id = metadatos.get("identifier", path.stem)
    titulo = metadatos.get("title", "Sin título")

    raw_status = metadatos.get("status", "unknown")

    # La fuente declara "in_force", pero la vigencia no fue
    # verificada contra registros oficiales.
    estado_mapeado = "unknown"

    contenido_limpio = apply("Espacios_Saltos", norm_spaces, contenido)

    # 5. Extraer secciones: artículos y anexos

    sections = []

    # Detectamos artículos y anexos por separado
    articulos = [
        (m.start(), "articulo", m.group(1))
        for m in regex_articulo.finditer(contenido_limpio)
    ]

    anexos = [
      (m.start(), "anexo", m.group(1) or m.group(2))
      for m in regex_anexo.finditer(contenido_limpio)
    ]

    # Ordenamos todas las secciones según su posición en el texto
    marcadores = sorted(articulos + anexos, key=lambda x: x[0])

    if marcadores:

        # Texto anterior al primer artículo o anexo
        if marcadores[0][0] > 0:
            sections.append(Section(
                section_id=f"{ley_id}#intro",
                title="Introducción y considerandos",
                level=1,
                start=0,
                end=marcadores[0][0]
            ))

        seen_ids = set()

        for i, (inicio, tipo, numero) in enumerate(marcadores):

            # El final de cada sección es el comienzo de la siguiente
            fin = (
                marcadores[i + 1][0]
                if i + 1 < len(marcadores)
                else len(contenido_limpio)
            )

            if tipo == "articulo":
                base_id = f"{ley_id}#ART-{numero}"
                titulo_seccion = f"Artículo {numero}"
                nivel = 2

            else:
                numero = numero.upper()
                base_id = f"{ley_id}#ANEXO-{numero}"
                titulo_seccion = f"Anexo {numero}"
                nivel = 2

            # Resolver encabezados repetidos sin perder secciones
            sec_id = base_id
            contador = 2

            while sec_id in seen_ids:
                sec_id = f"{base_id}-{contador}"
                contador += 1

            seen_ids.add(sec_id)

            sections.append(Section(
                section_id=sec_id,
                title=titulo_seccion,
                level=nivel,
                start=inicio,
                end=fin
            ))

    else:
        # Documento sin encabezados de artículo ni anexo reconocidos
        sections.append(Section(
            section_id=f"{ley_id}#unico",
            title="Texto completo",
            level=1,
            start=0,
            end=len(contenido_limpio)
        ))

    # 6. Armar el objeto Document oficial
    doc = Document(
        doc_id=ley_id,
        title=titulo,
        text=contenido_limpio,
        sections=sections,
        metadata=DocMetadata(
          date=metadatos.get("publication_date"),
          version=None,
          status=estado_mapeado,
          source_url=metadatos.get("source", ""),
          source_status=raw_status,
          lang="es"
        )
        )

    docs_finales.append(doc)




####



## ----------
import hashlib
from collections import Counter, defaultdict

def mejorar_ids_documento(doc):

    secciones = sorted(doc.sections, key=lambda s: s.start)

    candidatos = []
    anexo_actual = None

    # PASO 1: construir IDs usando el contexto del anexo
    for sec in secciones:

        if sec.title.startswith("Anexo "):

            # Recuperamos el nombre estructural del anexo
            numero = sec.title[len("Anexo "):].strip()
            base = f"{doc.doc_id}#ANEXO-{numero.upper()}"

            candidatos.append((sec, base))

            # Los artículos posteriores pertenecen provisionalmente
            # al último anexo detectado
            anexo_actual = base

        elif sec.title.startswith("Artículo "):

            numero = sec.title[len("Artículo "):].strip()

            if anexo_actual is None:
                base = f"{doc.doc_id}#ART-{numero}"
            else:
                base = f"{anexo_actual}/ART-{numero}"

            candidatos.append((sec, base))

        else:
            candidatos.append((sec, sec.section_id))

    # PASO 2: identificar los IDs que todavía se repiten
    frecuencias = Counter(base for _, base in candidatos)

    nuevas_secciones = []
    ids_usados = set()
    ambiguos = []

    for sec, base in candidatos:

        nuevo_id = base

        if frecuencias[base] > 1:

            # Huella del contenido de la sección
            contenido = doc.text[sec.start:sec.end]

            huella = hashlib.sha256(
                contenido.encode("utf-8")
            ).hexdigest()[:12]

            nuevo_id = f"{base}-H{huella}"

            ambiguos.append(base)

        # Protección adicional para contenido exactamente repetido
        contador = 2
        id_original = nuevo_id

        while nuevo_id in ids_usados:
            nuevo_id = f"{id_original}-{contador}"
            contador += 1

        ids_usados.add(nuevo_id)

        nuevas_secciones.append(
            sec.model_copy(
                update={"section_id": nuevo_id}
            )
        )

    nuevo_doc = doc.model_copy(
        update={"sections": nuevas_secciones}
    )

    return nuevo_doc, ambiguos


# Aplicamos la mejora a los 170 documentos
docs_mejorados = []
ids_ambiguos = []

for doc in docs_finales:

    nuevo_doc, ambiguos = mejorar_ids_documento(doc)

    docs_mejorados.append(nuevo_doc)
    ids_ambiguos.extend(ambiguos)

docs_finales = docs_mejorados

print("--- MEJORA DE IDENTIFICADORES ---")
print("Documentos procesados:", len(docs_finales))
print("Secciones:", sum(len(d.sections) for d in docs_finales))
print("Secciones con numeración ambigua:", len(ids_ambiguos))
## ---------


# Validación estructural de todos los documentos

errores = []
secciones_grandes = []
total_secciones = 0

for doc in docs_finales:

    secciones = sorted(doc.sections, key=lambda s: s.start)
    total_secciones += len(secciones)

    if not secciones:
        errores.append((doc.doc_id, "Sin secciones"))
        continue

    # Comprobar cobertura del texto
    if secciones[0].start != 0:
        errores.append((doc.doc_id, "El texto no comienza en offset 0"))

    if secciones[-1].end != len(doc.text):
        errores.append((doc.doc_id, "El texto no termina en el último offset"))

    # Comprobar continuidad entre secciones
    for anterior, siguiente in zip(secciones, secciones[1:]):
        if anterior.end != siguiente.start:
            errores.append(
                (doc.doc_id, "Hay huecos o superposiciones")
            )

    # Comprobar IDs únicos
    ids = [s.section_id for s in secciones]

    if len(ids) != len(set(ids)):
        errores.append((doc.doc_id, "IDs de sección duplicados"))

    # Identificar secciones extensas
    for sec in secciones:
        caracteres = sec.end - sec.start

        if caracteres > 30000:
            secciones_grandes.append(
                (doc.doc_id, sec.section_id, caracteres)
            )


print("--- VALIDACIÓN ESTRUCTURAL DEL CORPUS ---")
print(f"Documentos revisados: {len(docs_finales)}")
print(f"Secciones revisadas: {total_secciones}")
print(f"Errores estructurales: {len(errores)}")
print(f"Secciones mayores a 30.000 caracteres: {len(secciones_grandes)}")

print("\nErrores encontrados:")
for error in errores[:20]:
    print(error)

print("\nSecciones extensas (primeras 10):")
for sec in sorted(secciones_grandes, key=lambda x: x[2], reverse=True)[:10]:
    print(sec)


#####################



assert not errores, (
    f"Se encontraron {len(errores)} errores estructurales: "
    f"{errores[:5]}"
)

assert len(docs_finales) == 170, (
    f"Se esperaban 170 documentos y se obtuvieron {len(docs_finales)}"
)

assert all(doc.metadata.date is not None for doc in docs_finales), (
    "Hay documentos sin fecha de publicación"
)




####################



# 7. Guardar, generar Hash y Validar
docs_dicts = [d.model_dump(exclude_none=True) if hasattr(d, "model_dump") else d.dict(exclude_none=True) for d in docs_finales]
store.write_jsonl(ART / "documents.jsonl", docs_dicts)

corpus_sha = documents_hash(docs_finales)

print("--- RESULTADOS DE LA LIMPIEZA ---")
print(f"Impacto (Hits): {CLEANING_HITS}")
print(f"Caracteres eliminados: {CLEANING_CHARS}")
print(f"Hash del corpus: {corpus_sha}")

validate_documents(ART / "documents.jsonl").show().raise_if_failed()

run.note("documents", len(docs_finales))
run.finish()

== 01_documents · profile full · report -> /content/projecto_rag/artifacts/full/run_report.json
reusing legalize-ar @ 3d2450b5c09a
--- MEJORA DE IDENTIFICADORES ---
Documentos procesados: 170
Secciones: 7070
Secciones con numeración ambigua: 887
--- VALIDACIÓN ESTRUCTURAL DEL CORPUS ---
Documentos revisados: 170
Secciones revisadas: 7070
Errores estructurales: 0
Secciones mayores a 30.000 caracteres: 44

Errores encontrados:

Secciones extensas (primeras 10):
('DEC-50-2019', 'DEC-50-2019#ANEXO-II', 649672)
('DEC-764-2024', 'DEC-764-2024#ART-18', 242607)
('DEC-357-2002', 'DEC-357-2002#ANEXO-III', 242397)
('DEC-293-2024', 'DEC-293-2024#ART-10', 217483)
('DEC-1359-2004', 'DEC-1359-2004#ANEXO-II', 175880)
('DEC-493-2025', 'DEC-493-2025#ART-17', 160817)
('DEC-50-2019', 'DEC-50-2019#ANEXO-III', 158775)
('DEC-480-2022', 'DEC-480-2022#ANEXO-II-H2a980c0a0720', 147686)
('DEC-866-2025', 'DEC-866-2025#ART-25', 141988)
('DEC-660-1996', 'DEC-660-1996#ANEXO-II', 141389)
--- RESULTADOS DE LA LIMPIEZA 

{'stage': '01_documents',
 'profile': 'full',
 'config': {},
 'status': 'ok',
 'started_at': '2026-10-19 02:37:11',
 'notes': {'documents': 170},
 'llm': {},
 'wall_s': 15.3,
 'peak_vram_gb': 0.0,
 'finished_at': '2026-10-19 02:37:27'}

### Manifiesto

In [26]:
import json
import hashlib
from datetime import datetime, timezone

# Hash SHA-256 de un archivo
def sha256_archivo(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# Archivos incluidos en el corpus
archivos_manifest = []

for path in subconjunto:
    archivos_manifest.append({
        "path": str(path.relative_to(legal_root)),
        "sha256": sha256_archivo(path)
    })

# Construir manifiesto reproducible
manifest = {
    "corpus": "legalize-ar",
    "source_url": "https://github.com/legalize-dev/legalize-ar",
    "source_commit": "3d2450b5c09a",
    "selection_keywords": palabras_clave,
    "selected_documents": len(subconjunto),
    "documents_hash": corpus_sha,
    "normalization": [
        "Unicode NFC",
        "Reparación de caracteres de control",
        "Normalización de espacios y saltos de línea"
    ],
    "files": archivos_manifest
}

# Guardar
with open(ART / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifiesto generado correctamente")
print("Archivos registrados:", len(archivos_manifest))
print("Ubicación:", ART / "manifest.json")

Manifiesto generado correctamente
Archivos registrados: 170
Ubicación: /content/projecto_rag/artifacts/full/manifest.json


### Limpieza

In [28]:
import difflib

# Seleccionamos los 9 documentos con caracteres de control
# y un documento adicional con tablas Markdown.

muestra_ids = [
    "LEY-26994",
    "DEC-1-2016",
    "DEC-402-2015",
    "DEC-2136-2013",
    "DEC-861-2022",
    "DNU-585-2024",
    "DEC-2095-2009",
    "DEC-741-2015",
    "DNU-575-2018",
    "DEC-1070-2014",
]

paths_por_id = {p.stem: p for p in subconjunto}

reporte = [
    "# Evidencia de limpieza: 10 documentos",
    "",
    "Comparación entre el cuerpo original y el cuerpo normalizado.",
    "Se conservan las marcas Markdown y los enlaces.",
    ""
]

for doc_id in muestra_ids:
    path = paths_por_id[doc_id]

    texto_original = path.read_text(encoding="utf-8")

    # Extraer el cuerpo original sin modificar los caracteres
    if texto_original.startswith("---"):
        partes = texto_original.split("---", 2)
        cuerpo_original = partes[2] if len(partes) >= 3 else texto_original
    else:
        cuerpo_original = texto_original

    # Aplicar exactamente las reglas del extractor
    cuerpo_limpio = norm_spaces(
        reparar_caracteres_control(
            norm_nfc(cuerpo_original)
        )
    )

    diferencias = list(difflib.unified_diff(
        cuerpo_original.splitlines(),
        cuerpo_limpio.splitlines(),
        fromfile=f"{path.name} (antes)",
        tofile=f"{path.name} (después)",
        lineterm=""
    ))

    reporte.extend([
        f"## {path.name}",
        "",
        f"- Caracteres originales: {len(cuerpo_original)}",
        f"- Caracteres normalizados: {len(cuerpo_limpio)}",
        "",
        "```diff",
        "\n".join(diferencias[:120]) if diferencias else "Sin cambios.",
        "```",
        ""
    ])

ruta_reporte = ART / "cleaning_examples.md"
ruta_reporte.write_text("\n".join(reporte), encoding="utf-8")

print("Reporte generado correctamente")
print("Documentos comparados:", len(muestra_ids))
print("Ubicación:", ruta_reporte)

Reporte generado correctamente
Documentos comparados: 10
Ubicación: /content/projecto_rag/artifacts/full/cleaning_examples.md


### Casi duplicados

In [30]:
import hashlib
import re
from collections import defaultdict
from itertools import combinations

# 1. Duplicados exactos
grupos_hash = defaultdict(list)

for doc in docs_finales:
    h = hashlib.sha256(doc.text.encode("utf-8")).hexdigest()
    grupos_hash[h].append(doc.doc_id)

duplicados_exactos = [
    ids for ids in grupos_hash.values()
    if len(ids) > 1
]

# 2. Similitud Jaccard de shingles de 5 palabras
def shingles(texto, n=5):
    palabras = re.findall(r"\w+", texto.lower())
    return set(
        tuple(palabras[i:i+n])
        for i in range(len(palabras) - n + 1)
    )

representaciones = {
    doc.doc_id: shingles(doc.text)
    for doc in docs_finales
}

candidatos = []

for doc_a, doc_b in combinations(docs_finales, 2):

    conjunto_a = representaciones[doc_a.doc_id]
    conjunto_b = representaciones[doc_b.doc_id]

    union = len(conjunto_a | conjunto_b)

    similitud = (
        len(conjunto_a & conjunto_b) / union
        if union else 1.0
    )

    if similitud >= 0.80:
        candidatos.append({
            "doc_a": doc_a.doc_id,
            "doc_b": doc_b.doc_id,
            "jaccard": round(similitud, 4)
        })

candidatos.sort(
    key=lambda x: x["jaccard"],
    reverse=True
)

print("--- DUPLICADOS Y CASI-DUPLICADOS ---")
print("Documentos:", len(docs_finales))
print("Pares comparados:", len(docs_finales) * (len(docs_finales) - 1) // 2)
print("Duplicados exactos:", len(duplicados_exactos))
print("Candidatos con Jaccard >= 0.80:", len(candidatos))

print("\nPrimeros candidatos:")
for par in candidatos[:10]:
    print(par)

--- DUPLICADOS Y CASI-DUPLICADOS ---
Documentos: 170
Pares comparados: 14365
Duplicados exactos: 0
Candidatos con Jaccard >= 0.80: 2

Primeros candidatos:
{'doc_a': 'DNU-805-2025', 'doc_b': 'LEY-26221', 'jaccard': 0.8967}
{'doc_a': 'DEC-493-2025', 'doc_b': 'DNU-805-2025', 'jaccard': 0.8413}


In [31]:
import json

dedup_report = {
    "method": "exact_sha256_and_word_5_shingle_jaccard",
    "documents": len(docs_finales),
    "pairs_compared": len(docs_finales) * (len(docs_finales) - 1) // 2,
    "exact_duplicate_groups": duplicados_exactos,
    "near_duplicate_threshold": 0.80,
    "near_duplicate_candidates": candidatos,
    "policy": (
        "Se conservan todos los documentos. "
        "La similitud textual no demuestra equivalencia jurídica. "
        "Los candidatos requieren revisión antes de cualquier eliminación."
    )
}

ruta_dedup = ART / "dedup_report.json"

with open(ruta_dedup, "w", encoding="utf-8") as f:
    json.dump(dedup_report, f, ensure_ascii=False, indent=2)

print("Informe de duplicados generado correctamente")
print("Candidatos registrados:", len(candidatos))
print("Ubicación:", ruta_dedup)

Informe de duplicados generado correctamente
Candidatos registrados: 2
Ubicación: /content/projecto_rag/artifacts/full/dedup_report.json


##corpus_card

In [35]:
corpus_card = f"""# Corpus Card — RAG Jurídico Argentino

## 1. Descripción

Corpus de documentos normativos argentinos preparado para un sistema
de Retrieval-Augmented Generation (RAG).

## 2. Procedencia

- Repositorio: https://github.com/legalize-dev/legalize-ar
- Commit solicitado: 3d2450b5c09a
- Formato de origen: Markdown con metadatos YAML.
- Idioma: español.
- Documentos seleccionados: {len(docs_finales)}.

## 3. Criterio de selección

Se seleccionaron los archivos Markdown que contienen al menos una de
las siguientes expresiones, sin distinguir mayúsculas y minúsculas:

{", ".join(f"`{p}`" for p in palabras_clave)}

La búsqueda se realizó sobre el archivo completo, incluido su YAML.

## 4. Normalización

Se aplicaron las siguientes reglas:

- Normalización Unicode NFC.
- Sustitución de caracteres de control identificados.
- Normalización de espacios y saltos de línea.
- Extracción de metadatos YAML, conservando el cuerpo normativo.

Documentos modificados por regla: {CLEANING_HITS}

Variación neta de caracteres por regla: {CLEANING_CHARS}

Las variaciones corresponden a diferencias de longitud;
no necesariamente representan contenido jurídico eliminado.

## 5. Metadatos

Cada documento incluye identificador, título, texto, secciones y metadatos.

- Fecha de publicación disponible: {sum(d.metadata.date is not None for d in docs_finales)}/{len(docs_finales)}.
- Versiones verificadas: 0.
- Vigencia verificada: no.

El estado normalizado se establece como `unknown`.
El estado declarado por la fuente se conserva por separado en
`source_status`, sin asumir que acredita la vigencia jurídica actual.

## 6. Segmentación estructural

Se identifican encabezados de artículos y anexos mediante expresiones
regulares. El texto anterior al primer encabezado se conserva como
introducción.

- Documentos: {len(docs_finales)}.
- Secciones: {total_secciones}.
- Errores de cobertura, continuidad o unicidad detectados: {len(errores)}.
- Secciones superiores a 30.000 caracteres: {len(secciones_grandes)}.

Los identificadores de sección incorporan el contexto del anexo
cuando este puede reconocerse mediante encabezados.

Cuando una numeración se repite dentro del mismo contexto, se
utiliza una huella SHA-256 del contenido para distinguir secciones.

Se identificaron 887 secciones con numeración ambigua.

Este procedimiento mejora la trazabilidad y garantiza identificadores
únicos en el corpus procesado, pero no asegura estabilidad absoluta
ante modificaciones del contenido ni reconstruye toda la jerarquía
jurídica interna.

## 7. Duplicados y casi-duplicados

- Método para duplicados exactos: SHA-256 del texto normalizado.
- Método de similitud: Jaccard de conjuntos de secuencias de 5 palabras.
- Pares comparados: {len(docs_finales) * (len(docs_finales) - 1) // 2}.
- Grupos de duplicados exactos: {len(duplicados_exactos)}.
- Umbral para candidatos: 0,80.
- Candidatos encontrados: {len(candidatos)}.

{chr(10).join(f"- `{c['doc_a']}` / `{c['doc_b']}`: {c['jaccard']:.4f}" for c in candidatos)}

**Política:** se conservan los 170 documentos. Ningún documento se
elimina por similitud textual sin revisión jurídica.

## 8. Limitaciones

- Algunas normas contienen artículos de otras normas o numeraciones repetidas.
- La jerarquía de identificadores es parcial: identifica anexos,
  pero no todos los capítulos, subanexos o normas incorporadas.
- Los identificadores basados en hashes pueden cambiar cuando
  cambia el contenido o los límites de una sección.
- La pertenencia al último anexo detectado es una aproximación
  estructural que no implica atribución jurídica verificada.
- Los anexos pueden incluir contenido visual indicado como `[IMG]`,
  cuyo contenido no fue extraído.
- Algunas secciones son extensas y necesitarán subdivisión durante
  la etapa de chunking.
- La limpieza de espacios puede modificar el formato Markdown;
  las diferencias quedan registradas en `cleaning_examples.md`.
- No se verificó la vigencia normativa contra fuentes oficiales.
- Una similitud textual elevada no demuestra equivalencia jurídica.

## 9. Artefactos

- `documents.jsonl`: documentos normalizados y secciones.
- `manifest.json`: procedencia, selección y hashes de archivos.
- `cleaning_examples.md`: diez comparaciones antes/después.
- `dedup_report.json`: duplicados y candidatos a casi-duplicados.
- `corpus_card.md`: esta documentación.

## 10. Integridad

Hash del corpus procesado: `{corpus_sha}`.
"""

ruta_card = ART / "corpus_card.md"
ruta_card.write_text(corpus_card, encoding="utf-8")

print("Corpus card generado correctamente")
print("Ubicación:", ruta_card)

Corpus card generado correctamente
Ubicación: /content/projecto_rag/artifacts/full/corpus_card.md
